# Deepfake Hybrid Detection — Colab Training Notebook

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

### Dataset structures expected

**FaceForensics++ (FFPP)**
```
ffpp-dataset/
├── original_sequences/youtube/c23/videos/   ← real videos (.mp4)
└── manipulated_sequences/
    ├── Deepfakes/c23/videos/                ← fake videos (.mp4)
    └── Face2Face/c23/videos/               ← fake videos (.mp4)
```

**Celeb-DF v2 (CDF)**
```
celeb-df/
├── Celeb-real/          ← real celebrity videos (.mp4)
├── YouTube-real/        ← real YouTube videos (.mp4)
└── Celeb-synthesis/     ← fake synthesis videos (.mp4)
```

### Pipeline steps
1. Mount Google Drive & clone project code
2. Install dependencies
3. Download FFPP from server + unzip & subsample CDF from Drive
4. Update `config.yaml` paths
5. Process FFPP (extract frames → delete videos → splits → FFT cache)
6. Process CDF (extract frames → delete videos → splits → FFT cache)
7. Train models (on FFPP and/or CDF)
8. Evaluate & save outputs back to Drive
9. Save everything back to Drive

> **Disk space strategy:** Each dataset is processed sequentially — raw videos
> are deleted right after frame extraction to free disk before computing the
> FFT cache. This keeps peak disk usage manageable on Colab's ~107 GB SSD.

## Step 1 — Mount Drive & clone project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# ---------------------------------------------------------------------------
# OPTION A: your code is on GitHub — replace with your actual repo URL
# ---------------------------------------------------------------------------
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/skripsi

# ---------------------------------------------------------------------------
# OPTION B: your code is already in Google Drive
#   Change the path below to where your project folder lives in Drive
# ---------------------------------------------------------------------------
CODE_IN_DRIVE = "/content/drive/MyDrive/skripsi"   # <-- EDIT THIS
!cp -r "{CODE_IN_DRIVE}" /content/skripsi

os.chdir("/content/skripsi/deepfake_hybrid")
!pwd && ls

/content/skripsi/deepfake_hybrid
 colab_train.ipynb
 command.txt
 config.CDF.sampled.yaml
 config.FFPP.sampled.yaml
 config.yaml
 dataset
'Design Eksperimental Skripsi Giovanny - Kuantitatif.docx'
 outputs
 README.md
 requirements.txt
 scripts
 src


## Step 2 — Install dependencies

In [ ]:
!pip install -q timm scikit-learn tqdm opencv-python-headless
# ffmpeg is needed so OpenCV can decode H.264/H.265 .mp4 files
!apt-get install -qq ffmpeg libavcodec-extra
# torch + torchvision are pre-installed on Colab GPU runtimes
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Selecting previously unselected package libaribb24-0:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../libaribb24-0_1.0.3-2_amd64.deb ...
Unpacking libaribb24-0:amd64 (1.0.3-2) ...
Selecting previously unselected package libopencore-amrnb0:amd64.
Preparing to unpack .../libopencore-amrnb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrnb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libopencore-amrwb0:amd64.
Preparing to unpack .../libopencore-amrwb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrwb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libvo-amrwbenc0:amd64.
Preparing to unpack .../libvo-amrwbenc0_0.1.3-2_amd64.deb ...
Unpacking libvo-amrwbenc0:amd64 (0.1.3-2) ...
dpkg: libavcodec58:amd64: dependency problems, but removing anyway as you requested:
 libchromaprint1:amd64 depends on libavcodec58 (>= 7:4.4).
 libavformat58:amd64 depends on libavcodec58 (= 7:4.4.2-0ubuntu0.22.04.1).
 libavfilter7:a

## Step 3 — Download FFPP + unzip & subsample CDF

**FFPP** — Downloaded directly from the FaceForensics++ server using `download_datasets.py`.
Downloads all ~1000 videos per category (original, Deepfakes, Face2Face) at c23 compression.
This takes a while (~30-60 min depending on server speed).

**Celeb-DF v2 (CDF)** — Unzipped from Drive, then subsampled to ~1500 balanced videos
to keep disk usage reasonable. The full zip has ~5000+ videos which is overkill.

In [ ]:
import os

# ── CDF (zip) — verify it exists in Drive before we start ────────────────────
CDF_ZIP = "/content/drive/MyDrive/skripsi/deepfake_hybrid/dataset/celeb_df/Celeb-DF-v2.zip"  # <-- EDIT THIS if path differs

print("=== Pre-flight checks ===")
print()
print("FFPP: will be downloaded from FaceForensics++ server (no Drive copy needed)")
print()
if os.path.isfile(CDF_ZIP):
    size_gb = os.path.getsize(CDF_ZIP) / 1_073_741_824
    print(f"OK  [CDF zip found] {CDF_ZIP} — {size_gb:.2f} GB")
else:
    print(f"MISSING [CDF zip not found] {CDF_ZIP}")
    print("    Check that the path above matches where Celeb-DF-v2.zip lives in your Drive.")

# Show available disk space
print()
!df -h / | head -2

In [ ]:
import os, shutil, random
from pathlib import Path

CONFIG_PATH = "/content/skripsi/deepfake_hybrid/config.yaml"

# ── 3a: Download FFPP from server ────────────────────────────────────────────
# Downloads all ~1000 videos per category (original, Deepfakes, Face2Face)
# at c23 compression. The TOS prompt is auto-accepted via echo pipe.
print("=" * 60)
print("Downloading FFPP from FaceForensics++ server...")
print("This may take 30-60 minutes. Go grab a coffee.")
print("=" * 60)
!echo "" | python /content/skripsi/deepfake_hybrid/scripts/download_datasets.py \
    --config "{CONFIG_PATH}" \
    --datasets original Deepfakes Face2Face \
    --compression c23 \
    --type videos \
    --server EU2

# Verify download
print("\n=== FFPP download verification ===")
ffpp_root = Path("/content/ffpp")
for label, subpath in [
    ("real (original)", "original_sequences/youtube/c23/videos"),
    ("fake (Deepfakes)", "manipulated_sequences/Deepfakes/c23/videos"),
    ("fake (Face2Face)", "manipulated_sequences/Face2Face/c23/videos"),
]:
    p = ffpp_root / subpath
    if p.is_dir():
        count = len(list(p.glob("*.mp4")))
        print(f"  OK [{label}] — {count} .mp4 files")
    else:
        print(f"  MISSING [{label}] {p}")

# ── 3b: Unzip CDF + subsample to ~1500 videos ───────────────────────────────
print("\n" + "=" * 60)
print("Unzipping Celeb-DF-v2.zip from Drive...")
print("=" * 60)

CDF_FULL = "/content/celeb_df_full"
CDF_SAMPLED = "/content/celeb_df"
os.makedirs(CDF_FULL, exist_ok=True)
!unzip -q "{CDF_ZIP}" -d "{CDF_FULL}"

# Flatten wrapper subfolder if present (e.g. Celeb-DF-v2/)
celeb_full = Path(CDF_FULL)
subdirs = [d for d in celeb_full.iterdir() if d.is_dir()]
expected = {"Celeb-real", "YouTube-real", "Celeb-synthesis"}
if subdirs and not expected.intersection({d.name for d in subdirs}):
    wrapper = subdirs[0]
    print(f"Flattening wrapper folder '{wrapper.name}'...")
    for item in wrapper.iterdir():
        shutil.move(str(item), str(celeb_full / item.name))
    wrapper.rmdir()

# Count available videos
real_dirs = ["Celeb-real", "YouTube-real"]
fake_dirs = ["Celeb-synthesis"]
all_real = []
all_fake = []
for d in real_dirs:
    p = celeb_full / d
    if p.exists():
        all_real.extend(list(p.glob("*.mp4")))
for d in fake_dirs:
    p = celeb_full / d
    if p.exists():
        all_fake.extend(list(p.glob("*.mp4")))

print(f"\nFull CDF: {len(all_real)} real + {len(all_fake)} fake = {len(all_real) + len(all_fake)} total")

# Balanced subsample to ~1500 videos (750 real + 750 fake)
SAMPLE_SIZE = 1500
half = SAMPLE_SIZE // 2
rng = random.Random(42)
rng.shuffle(all_real)
rng.shuffle(all_fake)
real_take = min(len(all_real), half)
fake_take = min(len(all_fake), SAMPLE_SIZE - real_take)
selected_real = all_real[:real_take]
selected_fake = all_fake[:fake_take]

print(f"Sampling {len(selected_real)} real + {len(selected_fake)} fake = {len(selected_real) + len(selected_fake)} videos")

# Copy sampled videos preserving folder structure
os.makedirs(CDF_SAMPLED, exist_ok=True)
for src in selected_real + selected_fake:
    rel = src.relative_to(celeb_full)
    dst = Path(CDF_SAMPLED) / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# Delete the full extraction to free ~9 GB
shutil.rmtree(CDF_FULL)
print(f"Deleted {CDF_FULL} to free disk space.")

# Verify sampled CDF
print("\n=== Sampled CDF verification ===")
for subdir in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    p = Path(CDF_SAMPLED) / subdir
    if p.exists():
        mp4s = list(p.glob("*.mp4"))
        print(f"  {subdir}/  — {len(mp4s)} .mp4 files")
    else:
        print(f"  {subdir}/  — not present (may be expected if all samples came from other dirs)")

print()
!df -h / | head -2

## Step 4 — Update config.yaml paths

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = "/content/skripsi/deepfake_hybrid/config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ── FFPP ────────────────────────────────────────────────────────────────────
cfg['ffpp_root'] = "/content/ffpp"
cfg['datasets']['ffpp']['root'] = "/content/ffpp"
cfg['datasets']['ffpp']['real_keywords'] = ["original", "real", "pristine", "actors", "youtube"]
cfg['datasets']['ffpp']['fake_keywords'] = ["fake", "manipulated", "deepfakes", "faceswap",
                                             "neuraltextures", "deepfakedetection", "faceshifter",
                                             "face2face"]

# ── CDF ─────────────────────────────────────────────────────────────────────
# Celeb-DF v2 folder names: Celeb-real/, YouTube-real/ (real) and Celeb-synthesis/ (fake)
cfg['celebdf_root'] = "/content/celeb_df"
cfg['datasets']['cdf']['root'] = "/content/celeb_df"
cfg['datasets']['cdf']['real_keywords'] = ["real", "authentic", "youtube"]
cfg['datasets']['cdf']['fake_keywords'] = ["fake", "synthesis", "deepfake"]

# ── Shared settings ──────────────────────────────────────────────────────────
cfg['output_root'] = "/content/outputs"
cfg['epochs'] = 25           # 25 epochs with early stopping (patience=7)
cfg['n_seeds'] = 1           # increase to 3 for statistical validity
cfg['batch_size'] = 16       # reduce to 8 if you get OOM errors
cfg['num_workers'] = 2       # Colab works well with 2

# ── Training improvements ────────────────────────────────────────────────────
cfg['scheduler'] = 'cosine'           # cosine annealing with warmup
cfg['warmup_epochs'] = 2              # linear warmup for first 2 epochs
cfg['min_lr'] = 1e-6                  # minimum LR at end of cosine schedule
cfg['max_grad_norm'] = 1.0            # gradient clipping for stability
cfg['early_stopping_patience'] = 7    # stop if no AUC improvement for 7 epochs

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f)

print("Config updated:")
with open(CONFIG_PATH) as f:
    print(f.read())

## Step 5 — Process FFPP (frames → delete videos → splits → FFT cache)

Runs the full preprocessing pipeline for FFPP in one go.
Raw videos are deleted immediately after frame extraction to free disk space
before computing the FFT cache.

**If you get an empty manifest error**, run the debug cell below Step 6 to diagnose.

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

dataset = "FFPP"
local_root = "/content/ffpp"

# ── 5a: Extract frames ───────────────────────────────────────────────────────
print(f"{'='*50}")
print(f"Extracting frames: {dataset}")
print(f"{'='*50}")
!python /content/skripsi/deepfake_hybrid/scripts/extract_frames.py \
    --config "{CONFIG_PATH}" \
    --dataset "{dataset}" \
    --fps 5 \
    --max-frames 100

manifest_path = Path(f"/content/outputs/manifests/{dataset}/manifest.csv")
assert manifest_path.exists(), f"manifest.csv was not created for {dataset}"
manifest = pd.read_csv(manifest_path)
if len(manifest) == 0:
    raise RuntimeError(f"manifest.csv is empty for {dataset} — check {local_root}")
real_n = manifest['label'].eq(0).sum()
fake_n = manifest['label'].eq(1).sum()
print(f"  Total: {len(manifest)} | Real: {real_n} | Fake: {fake_n}")
if real_n == 0:
    raise RuntimeError(f"0 real videos in {dataset}")
if fake_n == 0:
    raise RuntimeError(f"0 fake videos in {dataset}")

# ── 5b: Delete raw FFPP videos to free disk ──────────────────────────────────
print(f"\nDeleting raw videos at {local_root} to free disk...")
shutil.rmtree(local_root)
print("Done.")
!df -h / | head -2

# ── 5c: Build splits ─────────────────────────────────────────────────────────
print(f"\nBuilding train/val/test splits for {dataset}...")
!python /content/skripsi/deepfake_hybrid/scripts/build_splits.py \
    --config "{CONFIG_PATH}" \
    --dataset "{dataset}" \
    --val-size 0.15 \
    --test-size 0.15 \
    --seed 42

for split in ['train', 'val', 'test']:
    df = pd.read_csv(f"/content/outputs/manifests/{dataset}/{split}.csv")
    print(f"  {split}: {len(df)} videos | real={df['label'].eq(0).sum()} fake={df['label'].eq(1).sum()}")

# ── 5d: Compute FFT cache ────────────────────────────────────────────────────
print(f"\nComputing FFT cache for {dataset}...")
!python /content/skripsi/deepfake_hybrid/scripts/compute_fft_cache.py \
    --config "{CONFIG_PATH}" \
    --dataset "{dataset}" \
    --num-workers 2
print(f"Done. Cache saved under /content/outputs/fft_cache/{dataset}/")
!df -h / | head -2

In [ ]:
# ── DEBUG (run this BEFORE Step 5/6 if they give an empty manifest) ──────────
# This checks whether .mp4 files exist and OpenCV can read them.
# Run it while the raw videos still exist (before they get deleted).
import cv2, glob

for ds, root in [("FFPP", "/content/ffpp"), ("CDF", "/content/celeb_df")]:
    print(f"\n=== {ds}: .mp4 files under {root} ===")
    mp4s = glob.glob(f"{root}/**/*.mp4", recursive=True)
    if mp4s:
        for p in mp4s[:5]:
            print(f"  {p}")
        cap = cv2.VideoCapture(mp4s[0])
        print(f"  OpenCV can open {mp4s[0]}: {cap.isOpened()}")
        cap.release()
    else:
        print(f"  No .mp4 files found — download/unzip in Step 3 may have failed")
# ─────────────────────────────────────────────────────────────────────────────

## Step 6 — Process CDF (frames → delete videos → splits → FFT cache)

Same sequential pipeline as Step 5, but for Celeb-DF.

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

dataset = "CDF"
local_root = "/content/celeb_df"

# ── 6a: Extract frames ───────────────────────────────────────────────────────
print(f"{'='*50}")
print(f"Extracting frames: {dataset}")
print(f"{'='*50}")
!python /content/skripsi/deepfake_hybrid/scripts/extract_frames.py \
    --config "{CONFIG_PATH}" \
    --dataset "{dataset}" \
    --fps 5 \
    --max-frames 100

manifest_path = Path(f"/content/outputs/manifests/{dataset}/manifest.csv")
assert manifest_path.exists(), f"manifest.csv was not created for {dataset}"
manifest = pd.read_csv(manifest_path)
if len(manifest) == 0:
    raise RuntimeError(f"manifest.csv is empty for {dataset} — check {local_root}")
real_n = manifest['label'].eq(0).sum()
fake_n = manifest['label'].eq(1).sum()
print(f"  Total: {len(manifest)} | Real: {real_n} | Fake: {fake_n}")
if real_n == 0:
    raise RuntimeError(f"0 real videos in {dataset}")
if fake_n == 0:
    raise RuntimeError(f"0 fake videos in {dataset}")

# ── 6b: Delete raw CDF videos to free disk ───────────────────────────────────
print(f"\nDeleting raw videos at {local_root} to free disk...")
shutil.rmtree(local_root)
print("Done.")
!df -h / | head -2

# ── 6c: Build splits ─────────────────────────────────────────────────────────
print(f"\nBuilding train/val/test splits for {dataset}...")
!python /content/skripsi/deepfake_hybrid/scripts/build_splits.py \
    --config "{CONFIG_PATH}" \
    --dataset "{dataset}" \
    --val-size 0.15 \
    --test-size 0.15 \
    --seed 42

for split in ['train', 'val', 'test']:
    df = pd.read_csv(f"/content/outputs/manifests/{dataset}/{split}.csv")
    print(f"  {split}: {len(df)} videos | real={df['label'].eq(0).sum()} fake={df['label'].eq(1).sum()}")

# ── 6d: Compute FFT cache ────────────────────────────────────────────────────
print(f"\nComputing FFT cache for {dataset}...")
!python /content/skripsi/deepfake_hybrid/scripts/compute_fft_cache.py \
    --config "{CONFIG_PATH}" \
    --dataset "{dataset}" \
    --num-workers 2
print(f"Done. Cache saved under /content/outputs/fft_cache/{dataset}/")
!df -h / | head -2

## Step 7 — Train

Each cell below trains one model type on one dataset.
Run all cells sequentially for the full 4-model × 2-dataset experiment,
or pick only the combinations you need.

In [ ]:
# --- Spatial-only (XceptionNet baseline) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training spatial on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model spatial \
        --seed 42 \
        --pretrained

In [ ]:
# --- Frequency-only (FreqCNN baseline) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training freq on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model freq \
        --seed 42

In [ ]:
# --- Hybrid two-branch (main model) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training hybrid on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model hybrid \
        --seed 42 \
        --pretrained

In [ ]:
# --- Early fusion (alternative fusion) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training early_fusion on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model early_fusion \
        --seed 42 \
        --pretrained

## Step 8 — Evaluate checkpoints

In [ ]:
from pathlib import Path

# Evaluate every checkpoint on the dataset it was trained on (in-dataset eval).
# The checkpoint naming convention is: {model}_{DATASET}_seed{seed}/best.pt
for dataset in ["FFPP", "CDF"]:
    for model_type in ['spatial', 'freq', 'hybrid', 'early_fusion']:
        ckpt = f"/content/outputs/runs/{model_type}_{dataset}_seed42/best.pt"
        if not Path(ckpt).exists():
            print(f"[SKIP] No checkpoint: {model_type} on {dataset}")
            continue
        print(f"\n=== Evaluating {model_type} trained on {dataset} ===")
        !python /content/skripsi/deepfake_hybrid/scripts/eval.py \
            --config "{CONFIG_PATH}" \
            --dataset "{dataset}" \
            --model "{model_type}" \
            --checkpoint "{ckpt}" \
            --seed 42 \
            --pretrained

In [ ]:
# Print a summary table of all results
import json
from pathlib import Path

results_dir = Path("/content/outputs/tables")
rows = []
for f in sorted(results_dir.glob("*.json")):
    data = json.loads(f.read_text())
    rows.append({"experiment": f.stem, **data})

if rows:
    import pandas as pd
    df = pd.DataFrame(rows).set_index("experiment")
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(df.to_string())
else:
    print("No results yet — run Step 9 first.")

## Step 9 — Save everything back to Drive

**Important:** Colab resets when the session ends. Always run this cell before closing.

In [ ]:
import shutil, os
from pathlib import Path
from datetime import datetime

# ── Where to save in your Drive ────────────────────────────────────────────
# A timestamped subfolder so each run doesn't overwrite the previous one.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_OUT = f"/content/drive/MyDrive/skripsi_outputs/{timestamp}"
os.makedirs(DRIVE_OUT, exist_ok=True)

LOCAL_OUT = "/content/outputs"

# ── 1. Model checkpoints (most important — save these first) ────────────────
runs_src = Path(LOCAL_OUT) / "runs"
if runs_src.exists():
    shutil.copytree(runs_src, f"{DRIVE_OUT}/runs")
    print(f"✓ Checkpoints saved → {DRIVE_OUT}/runs/")
else:
    print("⚠ No runs/ directory found — training may not have completed")

# ── 2. Results tables (JSON / CSV metrics) ──────────────────────────────────
tables_src = Path(LOCAL_OUT) / "tables"
if tables_src.exists():
    shutil.copytree(tables_src, f"{DRIVE_OUT}/tables")
    print(f"✓ Results saved   → {DRIVE_OUT}/tables/")

# ── 3. Manifests / splits ────────────────────────────────────────────────────
manifests_src = Path(LOCAL_OUT) / "manifests"
if manifests_src.exists():
    shutil.copytree(manifests_src, f"{DRIVE_OUT}/manifests")
    print(f"✓ Manifests saved → {DRIVE_OUT}/manifests/")

# ── 4. Updated config.yaml ───────────────────────────────────────────────────
config_src = Path("/content/skripsi/deepfake_hybrid/config.yaml")
if config_src.exists():
    shutil.copy2(config_src, f"{DRIVE_OUT}/config.yaml")
    print(f"✓ config.yaml saved → {DRIVE_OUT}/config.yaml")

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\nAll outputs saved to Drive folder:\n  {DRIVE_OUT}")
print("\nContents:")
for item in sorted(Path(DRIVE_OUT).rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / 1_048_576
        print(f"  {item.relative_to(DRIVE_OUT)}  ({size_mb:.1f} MB)")

✓ Checkpoints saved → /content/drive/MyDrive/skripsi_outputs/20260220_001723/runs/
✓ Results saved   → /content/drive/MyDrive/skripsi_outputs/20260220_001723/tables/
✓ Manifests saved → /content/drive/MyDrive/skripsi_outputs/20260220_001723/manifests/
✓ config.yaml saved → /content/drive/MyDrive/skripsi_outputs/20260220_001723/config.yaml

All outputs saved to Drive folder:
  /content/drive/MyDrive/skripsi_outputs/20260220_001723

Contents:
  config.yaml  (0.0 MB)
  manifests/FFPP/manifest.csv  (0.0 MB)
  manifests/FFPP/test.csv  (0.0 MB)
  manifests/FFPP/train.csv  (0.0 MB)
  manifests/FFPP/val.csv  (0.0 MB)
  runs/eval.log  (0.0 MB)
  runs/hybrid_FFPP_seed42/best.pt  (81.8 MB)
  runs/hybrid_FFPP_seed42/metrics.json  (0.0 MB)
  runs/hybrid_FFPP_seed42/train.log  (0.0 MB)
  tables/eval_hybrid_FFPP.json  (0.0 MB)


---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size` to `8` in Step 4 |
| `EmptyDataError: No columns to parse` in Step 5/6 | manifest.csv is empty — run the debug cell to see if `.mp4` files exist |
| `FileNotFoundError: manifest` in Step 5/6 | Frame extraction may have crashed — check output above |
| FFPP download stalls or times out | Try a different server: change `--server EU2` to `--server EU` or `--server CA` in Step 3 |
| Step 3 shows `MISSING [CDF zip not found]` | Update `CDF_ZIP` to the correct path of `Celeb-DF-v2.zip` in your Drive |
| Unzip fails with `End-of-central-directory` error | The zip in Drive is corrupted — re-upload it |
| After unzip, `Celeb-real/` etc. are MISSING | The zip extracted into a wrapper subfolder — the cell handles this automatically; if it fails, run `!ls /content/celeb_df_full` to see the actual layout |
| Debug cell shows `OpenCV can open ...: False` | Re-run Step 2 to install `ffmpeg`, then restart the runtime and re-run from Step 2 |
| Debug cell shows "No .mp4 files found" | Download or unzip in Step 3 failed — re-run Step 3 |
| FFPP: all videos have `unknown label` warning | Verify subfolders are exactly `original_sequences`, `Deepfakes`, `Face2Face` (case-sensitive) |
| CDF: all videos have `unknown label` warning | Verify top-level subfolders are exactly `Celeb-real`, `YouTube-real`, `Celeb-synthesis` (case-sensitive) |
| Disk full during FFT cache | Reduce `--max-frames` from 100 to 50 in Steps 5/6, or use fewer FFPP videos |
| Session disconnects mid-training | Outputs in `/content/` are lost — re-run Step 9 to save to Drive before closing |